# Readouts and losses

[`07a`](07a_native_classifier.ipynb) trained a classifier with two labels, a `sign` readout and the hinge loss. The readout and the loss decide what kind of problem a model solves. This notebook changes them to classify three classes, to predict real numbers and to find the lowest energy state of a Hamiltonian without any labels, and then compares training on exact probabilities with training on shots.

## Several classes: `argmax`

With more than two classes, the readout takes one observable per class, and `decision="argmax"` predicts the index of the one with the largest expected value. Here there are three classes, so three qubits with `⟨Z₀⟩`, `⟨Z₁⟩` and `⟨Z₂⟩`, and the labels are the class indices 0, 1 and 2. The ansatz is `hardware_efficient(1)`, another standard layout with `ry` and `rz` rotations on every qubit and a chain of `cx` gates.

The matching loss is `categorical_cross_entropy`: it turns the three expected values into probabilities and scores how much probability the right class gets. The optimizer is the `DE` of [`06a`](06a_variational_training.ipynb), which works on native models just as on circuits. The accuracy is measured on the same nine training samples, since the point here is the readout:

In [ ]:
import math

import numpy as np
import polypus
from polypus import qml

x_three = [
    [0.35, 0.40, 0.45],
    [0.45, 0.35, 0.40],
    [0.40, 0.45, 0.35],
    [1.50, 1.55, 1.60],
    [1.60, 1.50, 1.55],
    [1.55, 1.60, 1.50],
    [2.70, 2.75, 2.65],
    [2.65, 2.70, 2.75],
    [2.75, 2.65, 2.70],
]
y_three = [0, 0, 0, 1, 1, 1, 2, 2, 2]

three_classes = qml.Model(
    qml.Encoder(3).angle_encoder("ry"),
    qml.Ansatz(3).hardware_efficient(1),
    qml.Readout([qml.Z(0), qml.Z(1), qml.Z(2)], decision="argmax"),
)
result_three = three_classes.fit(
    x_three,
    y_three,
    # a tiny tolerance and patience=60: DE runs all 60 generations
    method=polypus.DE(generations=60, population_size=24, tolerance=1e-9, patience=60),
    loss="categorical_cross_entropy",
    exact=True,
    seed=17,
    id="three_classes",
)
predictions = three_classes.bind(result_three.best_params).predict(x_three, exact=True)
print("predictions:", predictions)
print("accuracy:", np.mean(np.array(predictions) == y_three), "| chance: 1/3")

## Regression: `raw`

A model can also predict a real number instead of a class. With `decision="raw"` the prediction is the expected value `⟨Z₀⟩` itself, a number between −1 and 1, and `loss="squared_error"` scores the squared distance to the target. Here the target is the curve 0.8·cos x, with one feature on a single qubit. The fitness is the mean squared error with its sign changed:

In [ ]:
import matplotlib.pyplot as plt

x_curve = [[0.2 + i * 0.25] for i in range(12)]
y_curve = [0.8 * math.cos(row[0]) for row in x_curve]

regression = qml.Model(
    qml.Encoder(1).angle_encoder("ry"),
    qml.Ansatz(1).hardware_efficient(1),
    qml.Readout([qml.Z(0)], decision="raw"),
)
result_curve = regression.fit(
    x_curve,
    y_curve,
    method=polypus.Adam(max_iters=200, learning_rate=0.05),
    loss="squared_error",
    exact=True,
    seed=3,
    id="regression",
)
fitted = regression.bind(result_curve.best_params).predict(x_curve, exact=True)
print("mean squared error:", -result_curve.best_fitness)
print("largest error:", max(abs(p - t) for p, t in zip(fitted, y_curve)))

points = [row[0] for row in x_curve]
plt.figure(figsize=(5, 3))
plt.plot(points, y_curve, "o", label="target 0.8·cos x")
plt.plot(points, fitted, "x", label="prediction")
plt.xlabel("x")
plt.legend()
plt.show()

## No labels: the ground state of a Hamiltonian

Without labels, `fit` maximizes the expected value of the readout's first observable: `loss="expectation"`, the default when `y` is missing. The readout itself becomes the objective, which turns the model into a variational eigensolver. To find the lowest energy of a Hamiltonian `H`, the readout measures `−H`, written as a `qml.Observable` with its coefficients negated, so that maximizing `⟨−H⟩` minimizes the energy `⟨H⟩`.

The Hamiltonian here is `H = Z₀Z₁ + 0.5·Z₁` on two qubits. Its lowest energy is −1.5, reached only in the state with qubit 1 in `1` and qubit 0 in `0`, the bitstring `10` with qubit 0 on the right as everywhere in the series. The single sample is the point `[0, 0]`: the `ry` encoder then rotates by nothing, so the ansatz alone prepares the state:

In [ ]:
minus_h = qml.Observable([(-1.0, qml.Z(0) @ qml.Z(1)), (-0.5, qml.Z(1))])
eigensolver = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).hardware_efficient(2),
    qml.Readout([minus_h]),
)
result_h = eigensolver.fit(
    [[0.0, 0.0]],
    method=polypus.Adam(max_iters=150, learning_rate=0.1, tolerance=1e-9, patience=150),
    exact=True,
    seed=4,
    id="ground_state",
)
print("energy found:", -result_h.best_fitness, "| lowest energy: -1.5")

`run` with `exact=True` returns an `ExactResult`, with the exact probability of each bitstring for each row. The most probable bitstring is the state the model prepared:

In [ ]:
state = eigensolver.bind(result_h.best_params).run([[0.0, 0.0]], exact=True)
probabilities = state.probabilities[0]
most_likely = max(probabilities, key=probabilities.get)
print("most probable state:", most_likely, "| probability", probabilities[most_likely])

## Exact or sampled

Every training so far used `exact=True`: the probabilities come straight from the statevector, without shot noise. That option needs the local native simulator, and its memory grows as 2ⁿ with the number of qubits `n`, so it is a simulation tool. A real quantum computer only gives shots, and a model trained on one estimates every fitness and every gradient from them. Without `exact=True`, `fit` trains that way, with `shots` measurements per circuit and the `seed` fixing the sampling, so the run still repeats exactly.

The same model is trained both ways below, on two clusters like those of [`07a`](07a_native_classifier.ipynb) written out as eight points. With shots, `best_fitness` is itself an estimate, so the parameters of both runs are scored again afterwards with `exact=True`:

In [ ]:
x_two = [
    [0.50, 0.55],
    [0.60, 0.45],
    [0.45, 0.60],
    [0.55, 0.50],
    [2.55, 2.60],
    [2.65, 2.50],
    [2.50, 2.65],
    [2.60, 2.55],
]
y_two = [-1, -1, -1, -1, 1, 1, 1, 1]

classifier = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).hardware_efficient(1),
    qml.Readout([qml.Z(0)], decision="sign"),
)
adam = polypus.Adam(max_iters=40, learning_rate=0.2)
exact_result = classifier.fit(
    x_two, y_two, method=adam, loss="hinge", exact=True, seed=5, id="exact"
)
shots_result = classifier.fit(
    x_two, y_two, method=adam, loss="hinge", shots=1024, seed=5, id="shots"
)

for name, result in (("exact", exact_result), ("1024 shots", shots_result)):
    trained = classifier.bind(result.best_params)
    accuracy = np.mean(np.array(trained.predict(x_two, exact=True)) == y_two)
    print(f"{name:>10}: best_fitness {result.best_fitness:.4f}, accuracy {accuracy}")

A trained model also runs both ways. Without `exact=True`, `run` returns a `RunResult` with the counts measured for each row, the same kind of result as `run_quantum_circuit` in [`03a`](03a_local_execution.ipynb). With it, `run` returns an `ExactResult` with the probabilities. Divided by the number of shots, the counts come close to those probabilities:

In [ ]:
trained = classifier.bind(exact_result.best_params)
sample = x_two[:1]
counts = trained.run(sample, shots=1000, seed=3).counts[0]
probabilities = trained.run(sample, exact=True).probabilities[0]
for bits in sorted(probabilities):
    print(
        bits,
        "| counts / 1000:",
        counts.get(bits, 0) / 1000,
        "| exact:",
        round(probabilities[bits], 3),
    )

## Summary

This notebook changed the readout and the loss of a native model: `argmax` with `categorical_cross_entropy` for several classes, `raw` with `squared_error` for regression, and a readout without labels that finds the ground state of a Hamiltonian. It also trained the same model on exact probabilities and on shots, and ran it both ways with `run`.

## End of the series

This series has walked the full path: from the first circuit to training models with data. Readers who want to go further can check [`examples/`](https://github.com/Bahia-Software/polypus/tree/main/examples), with bigger cases, including a full MaxCut with QAOA example. Two of them continue this section: `qml_iris_demo.py` trains a native classifier on the Iris flowers, and `qml_native_showcase.py` goes through every native encoder, layer, optimizer, readout and loss. The [user guide](https://github.com/Bahia-Software/polypus/tree/main/docs/guide) covers training, QML, memory and performance in depth.

Back to the index: [`00_welcome.ipynb`](00_welcome.ipynb).